# Carleton evacuation — MARS

Set `SCENARIO` (1–7) and `RUN_SIMULATION` below, then run all cells.

| # | Scenario |
|---|----------|
| 1 | Baseline |
| 2 | P6 +1 h |
| 3 | P6 +2 h |
| 4 | P3 +1 h, P6 +2 h |
| 5 | P3/P4 +1 h, P6 +2 h |
| 6 | P3/P4 +1 h, P6 +1.5 h |
| 7 | Baseline timing; P3/P4 emergency exit |

**Step 1** — Load the project folder and helpers for running commands.

In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import sys
from pathlib import Path

from IPython.display import Image, Markdown, display

ROOT = Path.cwd()
if not (ROOT / "SOHCarletonDrivingBox.csproj").is_file():
    candidate = ROOT / "CarletonDrivingBox"
    if (candidate / "SOHCarletonDrivingBox.csproj").is_file():
        ROOT = candidate

SCRIPTS = ROOT / "scripts"
RESULTS = ROOT / "results"
CONFIGS = ROOT / "configs"
PROJECT = "SOHCarletonDrivingBox.csproj"

sys.path.insert(0, str(SCRIPTS))
from mars_agent_outputs import agent_output_path


def run_cmd(cmd: list[str], *, quiet: bool = False, label: str | None = None) -> None:
    if label:
        print(label, flush=True)
    elif not quiet:
        print("Running:", " ".join(cmd), flush=True)
    completed = subprocess.run(
        cmd,
        cwd=ROOT,
        env={**os.environ, "DOTNET_NOLOGO": "1"},
        capture_output=quiet,
        text=True,
    )
    if completed.returncode != 0:
        details = ""
        if quiet:
            details = "\n" + (completed.stdout or "") + (completed.stderr or "")
        raise RuntimeError(
            f"Command failed ({completed.returncode}): {' '.join(cmd)}{details}"
        )
    if quiet:
        print("  done.", flush=True)


assert (ROOT / PROJECT).is_file(), f"Open this notebook from {ROOT.name}/"

**Step 2** — Pick the scenario and whether to run a new simulation or reuse existing results.

In [ ]:
SCENARIO = 7
RUN_SIMULATION = False
BUILD_FIRST = True
RUN_HEATMAP = True

SID = f"{SCENARIO:02d}"
CONFIG = CONFIGS / f"config_scenario_{SID}.json"
OUT_DIR = RESULTS / f"scenario_{SID}"
CSV_PATH = agent_output_path(OUT_DIR, ".csv")

assert CONFIG.is_file(), f"Missing config: {CONFIG}"
print(f"scenario_{SID}  |  run={RUN_SIMULATION}  |  results: {OUT_DIR.relative_to(ROOT)}/")

**Step 3** — Build and run the MARS simulation (skipped when `RUN_SIMULATION` is False).

In [ ]:
if RUN_SIMULATION:
    if BUILD_FIRST:
        run_cmd(
            ["dotnet", "build", PROJECT, "-v", "q", "/p:NuGetAudit=false", "/clp:ErrorsOnly"],
            quiet=True,
            label="Building…",
        )
    cmd = ["dotnet", "run", "--no-launch-profile", "--project", PROJECT]
    if BUILD_FIRST:
        cmd.insert(2, "--no-build")
    cmd += ["--", str(CONFIG.relative_to(ROOT))]
    run_cmd(cmd, label="Running simulation…")
else:
    print("Simulation skipped.")

**Step 4** — Analyze the CSV and show summary, evacuation curve, and heatmap charts.

In [ ]:
if not CSV_PATH.is_file():
    raise FileNotFoundError(f"No CSV at {CSV_PATH}. Set RUN_SIMULATION = True and re-run.")

if RUN_HEATMAP:
    run_cmd([sys.executable, str(SCRIPTS / "analyze_and_heatmap.py"), str(CSV_PATH)])
else:
    run_cmd([sys.executable, str(SCRIPTS / "analyze_run.py"), str(CSV_PATH)])

summary_csv = OUT_DIR / "summary.csv"
if summary_csv.is_file():
    display(Markdown("### Summary"))
    print(summary_csv.read_text(encoding="utf-8"))

for name, title in {
    "summary.png": "### Deployment",
    "evac_curve.png": "### Evacuation curve",
    "heatmap_matrix.png": "### Heatmap",
}.items():
    path = OUT_DIR / name
    if path.is_file():
        display(Markdown(title))
        display(Image(filename=str(path)))